# Day 2: データパイプライン & マスキング

| # | セクション | 学ぶこと | 時間 |
|---|-----------|----------|------|
| 1 | データ取り込み | 外部Stage、COPY INTO | 10分 |
| 2 | 半構造化データ | VARIANT型、FLATTEN | 10分 |
| 3 | Dynamic Table | 宣言的パイプライン、自動リフレッシュ | 10分 |
| 4 | RBAC | ロールベースアクセスコントロール | 10分 |
| 5 | マスキングポリシー | カラムレベルセキュリティ | 10分 |

---

> **前提条件**: `setup_handson.sql` が実行済みであること

## 事前準備：セッションコンテキストの設定

In [ ]:
%%sql -r dataframe_1
USE DATABASE tb_101;
USE ROLE tb_data_engineer;
USE WAREHOUSE tb_de_wh;

---
## 1. 外部Stageからのデータ取り込み

Snowflakeにおける**ステージ**とは、データファイルの保存場所を指定する名前付きオブジェクトです。

| 概念 | 説明 |
|------|------|
| 外部ステージ | S3/Azure Blob/GCSのパスを指す |
| ファイルフォーマット | CSV/JSON/Parquet等のパース方法を定義 |
| COPY INTO | ステージからテーブルにデータをバルクロード |

### Step 1: メニューデータ用のテーブルを作成

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE raw_pos.menu_staging
(
    menu_id NUMBER(19,0),
    menu_type_id NUMBER(38,0),
    menu_type VARCHAR(16777216),
    truck_brand_name VARCHAR(16777216),
    menu_item_id NUMBER(38,0),
    menu_item_name VARCHAR(16777216),
    item_category VARCHAR(16777216),
    item_subcategory VARCHAR(16777216),
    cost_of_goods_usd NUMBER(38,4),
    sale_price_usd NUMBER(38,4),
    menu_item_health_metrics_obj VARIANT
);

### Step 2: AWS S3バケットのステージ作成

このステップは **Snowsight GUI** でも作成できます：

1. 左メニュー → **カタログ** → **データベースエクスプローラー** → **TB_101** → **RAW_POS**
2. 右上の **作成** → **ステージ** → **Amazon S3** を選択
3. 以下を入力：

| 項目 | 値 |
|------|-----|
| Stage Name | `MENU_STAGE` |
| Comment | `Stage for menu data` |
| URL | `s3://sfquickstarts/frostbyte_tastybytes/raw_pos/menu/` |
| 認証 | 空欄（パブリックS3のため不要） |
| ディレクトリテーブル | チェックあり |
| 暗号化 | 空欄 |

4. **Create** をクリック

**注意**: SQLで実行する場合は下のセルを使用してください。

In [ ]:
%%sql -r dataframe_27
--クエリの場合はこちら
CREATE OR REPLACE STAGE raw_pos.menu_stage
COMMENT = 'Stage for menu data'
URL = 's3://sfquickstarts/frostbyte_tastybytes/raw_pos/menu/'
FILE_FORMAT = public.csv_ff;

### Step 3: COPY INTO でデータをロード

このステップは **Snowsight GUI** でも実行できます：

1. 左メニュー → **カタログ** → **データベースエクスプローラー** → **TB_101** → **RAW_POS**
2. 右上の **作成** → **テーブル** → **ファイルから** ボタンをクリック
3. データソースとして **ステージから追加** を選択
4. ステージ一覧から **TB101.RAW_POS.MENU_STAGE** を選択
5. ファイル `menu.csv.gz` が表示されるのでチェックを入れる
6. テーブルの選択において**MENU_STAGING**を選択し、**次へ**を選択
6. ファイル形式は既にステージに紐付いた `CSV_FF` が適用される
7. **ロード** をクリック


**注意**: SQLで実行する場合は下のセルを使用してください。

In [ ]:
%%sql -r dataframe_3
COPY INTO raw_pos.menu_staging
FROM @raw_pos.menu_stage;

### Step 4: ロード結果の確認

In [ ]:
%%sql -r dataframe_4
SELECT * FROM raw_pos.menu_staging;

---
## 2. 半構造化データとVARIANT型

SnowflakeはVARIANT型でJSON等の半構造化データを直接格納・クエリできます。

| 構文 | 用途 |
|------|------|
| `:` (コロン) | JSONキーへのアクセス |
| `[]` (角括弧) | 配列の要素アクセス |
| `::型` | VARIANTからのキャスト |
| `FLATTEN` | 配列/オブジェクトを行に展開 |

### Step 1: VARIANTカラムの中身を確認

In [ ]:
%%sql -r dataframe_5
SELECT menu_item_health_metrics_obj
FROM raw_pos.menu_staging
LIMIT 3;

### Step 2: コロン演算子でデータを抽出

In [ ]:
%%sql -r dataframe_6
SELECT
    menu_item_name,
    menu_item_health_metrics_obj:menu_item_id::INTEGER AS menu_item_id,
    menu_item_health_metrics_obj:menu_item_health_metrics[0]:ingredients::ARRAY AS ingredients
FROM raw_pos.menu_staging
LIMIT 10;

### Step 3: FLATTEN で配列を行に展開

`LATERAL FLATTEN` は配列の各要素を個別の行として展開します。

In [ ]:
%%sql -r dataframe_7
SELECT
    i.value::STRING AS ingredient_name,
    m.menu_item_health_metrics_obj:menu_item_id::INTEGER AS menu_item_id
FROM
    raw_pos.menu_staging m,
    LATERAL FLATTEN(INPUT => m.menu_item_health_metrics_obj:menu_item_health_metrics[0]:ingredients::ARRAY) i
LIMIT 20;

---
## 3. Dynamic Table

Dynamic Tableはデータ変換パイプラインを簡素化する強力なツールです。

| 特徴 | 説明 |
|------|------|
| 宣言的構文 | SELECTクエリでデータを定義 |
| 自動リフレッシュ | ソースデータの変更を自動検出・反映 |
| LAG設定 | データ鮮度の目標を指定（例: 1分以内） |

### Step 1: 材料マスターのDynamic Tableを作成

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE DYNAMIC TABLE harmonized.ingredient
    LAG = '1 minute'
    WAREHOUSE = 'TB_DE_WH'
AS
SELECT
    ingredient_name,
    menu_ids
FROM (
    SELECT DISTINCT
        i.value::STRING AS ingredient_name,
        ARRAY_AGG(m.menu_item_id) AS menu_ids
    FROM
        raw_pos.menu_staging m,
        LATERAL FLATTEN(INPUT => menu_item_health_metrics_obj:menu_item_health_metrics[0]:ingredients::ARRAY) i
    GROUP BY i.value::STRING
);

In [ ]:
%%sql -r dataframe_9
SELECT * FROM harmonized.ingredient LIMIT 10;

### Step 2: 新メニューを追加して自動リフレッシュを確認

サンドイッチトラック「Better Off Bread」が新メニュー「バインミー」を導入しました。

In [ ]:
%%sql -r dataframe_10
INSERT INTO raw_pos.menu_staging
SELECT
    10101, 15, 'Sandwiches', 'Better Off Bread',
    157, 'Banh Mi', 'Main', 'Cold Option',
    9.0, 12.0,
    PARSE_JSON('{
      "menu_item_health_metrics": [{
        "ingredients": ["French Baguette", "Mayonnaise", "Pickled Daikon", "Cucumber", "Pork Belly"],
        "is_dairy_free_flag": "N",
        "is_gluten_free_flag": "N",
        "is_healthy_flag": "Y",
        "is_nut_free_flag": "Y"
      }],
      "menu_item_id": 157
    }');

### Step 3: リフレッシュの確認

> 「Query produced no results」と表示される場合は、Dynamic Tableがまだリフレッシュされていません。最大1分お待ちください。

In [ ]:
%%sql -r dataframe_11
SELECT * FROM harmonized.ingredient
WHERE ingredient_name IN ('French Baguette', 'Pickled Daikon');

> **DAG確認**: Snowsight左メニュー → Data → TB_101 → HARMONIZED → Dynamic Tables → INGREDIENT をクリックするとパイプラインDAGを可視化できます。

---
## 4. ロールベースアクセスコントロール（RBAC）

Snowflakeではロール（役割）単位でアクセス権限を管理します。ここでは以下のシナリオを体験します：

1. **カスタムロールの作成** — 分析担当者用のロールを作る
2. **権限なしで実行 → 失敗** — ロールに権限がないとデータにアクセスできないことを確認
3. **権限を付与 → 成功** — 必要な権限を付与してアクセスできるようになることを確認

### Step 1: 顧客データの確認

まず管理者ロールで顧客データを確認します。個人情報（名前、メール、電話番号など）が含まれています。

In [ ]:
%%sql -r dataframe_12
USE ROLE tb_admin;

SELECT TOP 10
    customer_id,
    first_name,
    last_name,
    e_mail,
    phone_number,
    birthday_date
FROM raw_customer.customer_loyalty;

### Step 2: 分析担当者用カスタムロールの作成

分析チーム用のロール `tb_analyst` を作成し、自分のユーザーに割り当てます。

In [ ]:
%%sql -r dataframe_13
USE ROLE useradmin;

CREATE OR REPLACE ROLE tb_analyst
    COMMENT = 'Analyst role for customer data analysis';

SET my_user = CURRENT_USER();
GRANT ROLE tb_analyst TO USER IDENTIFIER($my_user);

### Step 3: 権限なしでクエリ実行 → 失敗

`tb_analyst` ロールにはまだ何の権限も付与していません。このままクエリを実行するとエラーになります。

In [ ]:
%%sql -r dataframe_14
USE SECONDARY ROLES NONE;
USE ROLE tb_analyst;

SELECT TOP 10 * FROM tb_101.raw_customer.customer_loyalty;

> **↑ エラーが出ることを確認してください。** ウェアハウス、データベース、スキーマ、テーブルの権限がないためアクセスできません。

### Step 4: 必要な権限を付与

`tb_analyst` ロールにウェアハウス、データベース、スキーマ、テーブルへのアクセス権限を付与します。

In [ ]:
%%sql -r dataframe_15
USE ROLE securityadmin;

GRANT OPERATE, USAGE ON WAREHOUSE tb_dev_wh TO ROLE tb_analyst;
GRANT USAGE ON DATABASE tb_101 TO ROLE tb_analyst;
GRANT USAGE ON SCHEMA tb_101.raw_customer TO ROLE tb_analyst;
GRANT SELECT ON ALL TABLES IN SCHEMA tb_101.raw_customer TO ROLE tb_analyst;
GRANT USAGE ON SCHEMA tb_101.governance TO ROLE tb_analyst;

### Step 5: 権限付与後にクエリ実行 → 成功

今度は `tb_analyst` ロールでもデータにアクセスできるようになります。

In [ ]:
%%sql -r dataframe_16
USE ROLE tb_analyst;
USE WAREHOUSE tb_dev_wh;

SELECT TOP 10
    customer_id,
    first_name,
    last_name,
    e_mail,
    phone_number,
    birthday_date
FROM raw_customer.customer_loyalty;

> **↑ 今度はデータが表示されることを確認してください。** 権限を付与することで、同じロールでもアクセスが可能になりました。

---
## 5. マスキングポリシー

マスキングポリシーは**カラムレベルセキュリティ**を実現します。同じテーブルでも、ロールに応じて見え方が変わります。

| 仕組み | 説明 |
|--------|------|
| ポリシー定義 | CASE式でロールに応じた表示/マスクを制御 |
| カラムに適用 | 特定カラムにポリシーを直接適用 |
| 透過的 | アプリケーション変更不要、クエリ時に動的適用 |

ここでは `tb_admin` は全データ閲覧可、`tb_analyst` はPII項目がマスクされるシナリオを作ります。

### Step 1: STRING型用マスキングポリシーの作成

`tb_admin` と `ACCOUNTADMIN` はそのまま表示、それ以外のロールはマスクされます。

In [ ]:
%%sql -r dataframe_17
USE ROLE tb_admin;

CREATE OR REPLACE MASKING POLICY governance.mask_string_pii
  AS (original_value STRING)
  RETURNS STRING ->
    CASE
      WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'TB_ADMIN')
        THEN original_value
      ELSE '****MASKED****'
    END;

### Step 2: DATE型用マスキングポリシーの作成

In [ ]:
%%sql -r dataframe_18
CREATE OR REPLACE MASKING POLICY governance.mask_date_pii
  AS (original_value DATE)
  RETURNS DATE ->
    CASE
      WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'TB_ADMIN')
        THEN original_value
      ELSE DATE_TRUNC('year', original_value)
    END;

### Step 3: マスキングポリシーをカラムに適用

個人情報を含むカラムにマスキングポリシーを直接適用します。

In [ ]:
%%sql -r dataframe_19
ALTER TABLE raw_customer.customer_loyalty MODIFY
    COLUMN first_name SET MASKING POLICY governance.mask_string_pii,
    COLUMN last_name SET MASKING POLICY governance.mask_string_pii,
    COLUMN e_mail SET MASKING POLICY governance.mask_string_pii,
    COLUMN phone_number SET MASKING POLICY governance.mask_string_pii,
    COLUMN birthday_date SET MASKING POLICY governance.mask_date_pii;

### Step 4: tb_analyst ロールで確認 → マスクされる

分析担当者ロールでクエリすると、個人情報がマスクされます。

In [ ]:
%%sql -r dataframe_20
USE ROLE tb_analyst;

SELECT TOP 10
    customer_id,
    first_name,
    last_name,
    e_mail,
    phone_number,
    birthday_date
FROM raw_customer.customer_loyalty;

> **↑ `first_name`, `last_name`, `e_mail`, `phone_number` が `****MASKED****` に、`birthday_date` が年の初日になっていることを確認してください。**

### Step 5: tb_admin ロールで確認 → マスクされない

管理者ロールでは同じクエリでも全データがそのまま見えます。

In [ ]:
%%sql -r dataframe_21
USE ROLE tb_admin;

SELECT TOP 10
    customer_id,
    first_name,
    last_name,
    e_mail,
    phone_number,
    birthday_date
FROM raw_customer.customer_loyalty;

> **↑ 同じテーブル・同じクエリでも、ロールによって見え方が異なることがRBACとマスキングポリシーの組み合わせです。**

---
## まとめ

| 学んだこと | ポイント |
|-----------|--------|
| 外部Stage + COPY INTO | S3からのバルクロード、ファイルフォーマット指定 |
| VARIANT / FLATTEN | JSONを直接格納・展開・クエリ |
| Dynamic Table | 宣言的パイプライン、自動リフレッシュ |
| RBAC | ロール作成 → 権限なしで失敗 → 権限付与で成功 |
| マスキングポリシー | ロールに応じた動的マスキング、カラムに直接適用 |

### ストーリー全体の流れ

```
S3 → Stage → COPY INTO → テーブル → Dynamic Table
                                  ↓
                           ロール作成 → 権限付与 → アクセス制御
                                  ↓
                           マスキングポリシー → ロール別の表示制御
```